In [1]:
# 01 | Imports & project setup
import os
import sys
import json
import math
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import trimesh
import open3d as o3d
import ollama
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display
from sklearn.cluster import DBSCAN

# Run from the project root or its experiments/ directory.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
if not (PROJECT_ROOT / "utils.py").is_file():
    raise RuntimeError(f"Project root not found: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ObjaverseDataset import ObjaverseDataset
from utils import render_n_views, _remove_platform_only


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
# 02 | Load base object (Objaverse)
CATEGORY = "duck"
DATASET_LIMIT = 1
OBJECT_INDEX = 0

objaverse_dataset = ObjaverseDataset(category=CATEGORY, limit=DATASET_LIMIT)
current_object = objaverse_dataset[OBJECT_INDEX]
uid_value = current_object["uid"]
uid = uid_value[0] if isinstance(uid_value, (list, tuple)) else uid_value
mesh = current_object["mesh"]
if isinstance(mesh, (list, tuple)):
    mesh = mesh[0]
print(f"Loaded {CATEGORY} | UID: {uid}")

mesh.show()


dict_keys(['Band_Aid', 'Bible', 'CD_player', 'Christmas_tree', 'Dixie_cup', 'Ferris_wheel', 'Lego', 'Rollerblade', 'Sharpie', 'Tabasco_sauce', 'aerosol_can', 'air_conditioner', 'airplane', 'alarm_clock', 'alcohol', 'alligator', 'almond', 'ambulance', 'amplifier', 'anklet', 'antenna', 'apple', 'apricot', 'apron', 'aquarium', 'arctic_(type_of_shoe)', 'armband', 'armchair', 'armoire', 'armor', 'army_tank', 'artichoke', 'ashtray', 'asparagus', 'atomizer', 'automatic_washer', 'avocado', 'award', 'awning', 'ax', 'baboon', 'baby_buggy', 'backpack', 'bagel', 'baguet', 'bait', 'ball', 'ballet_skirt', 'balloon', 'bamboo', 'banana', 'bandage', 'bandanna', 'banjo', 'banner', 'barbell', 'barge', 'barrel', 'barrow', 'baseball', 'baseball_bat', 'baseball_cap', 'baseball_glove', 'basket', 'basketball', 'basketball_backboard', 'bass_horn', 'bat_(animal)', 'bath_mat', 'bath_towel', 'bathrobe', 'bathtub', 'battery', 'beachball', 'bead', 'beanbag', 'beanie', 'bear', 'bed', 'bedpan', 'bedspread', 'beef_(fo

In [4]:
# 03 | Tool inputs & experiment configuration
# Change this cell for a new insertion request; change cell 02 for a new base object.
COMPONENT_PROMPT = "car wheel"
DIRECTIONS = ["left"]
USER_PROMPT = """Attach a single functional car wheel below the left wing of the duck,
slightly toward its chest, on the lower body.
The wheel diameter should be approximately 10% of the duck's total body length.
The wheel must be upright; its plane should be parallel to the longitudinal axis
of the duck, and its axle should point sideways, perpendicular to the forward
direction. The outer rim should face outward.
The wheel should be naturally connected to the body, without floating or excessive intersection."""

# Existing generated component (FLUX / Pixal3D generation is outside this tool).
NEW_OBJECT_MESH_PATH = Path("generated_models/ComfyUI_00032.glb")
COMPONENT_IMAGE_PATH = Path("generated_images/Flux2_dev_00012_.png")  # optional provenance only
OLLAMA_HOST = "192.168.178.23"
MODEL_NAME = "qwen3-vl:8b-instruct"
FITTING_MODEL = MODEL_NAME  # may be changed to qwen3-vl:32b-instruct
SELECTION_MODE = "selection"  # selection | ranking
SEED = 42

# Rendering / grounding
N_BASE_VIEWS = 24
N_PART_VIEWS = 6
N_REFERENCE_VIEWS = 3
BASE_IMAGE_SIZE = 1008
FRAME_SCALE = 1.8
PART_FRAME_SCALE = 1.2
VIEW_THRESHOLD = 0.3
MAX_ANCHOR_VIEWS = 6
ANCHOR_MIN_SAMPLES = 3
ANCHOR_CLUSTER_RADIUS = 0.2
MARKER_RADIUS_RATIO = 0.025
ANCHOR_VISIBILITY_RADIUS = 30
REMOVE_BASE_PLATFORM = True

# Voxel voting and initial transform
VOXEL_RESOLUTION = 64
VOXEL_PADDING = 0.3
VOXEL_THRESHOLD = 0.8
PENETRATION_RATIO = 0.08

# Mesh fitting
N_FITTING_VIEWS = 3
FITTING_IMAGE_SIZE = 512
FITTING_FRAME_SCALE = FRAME_SCALE
N_SCALE_CANDIDATES = 3
N_SCALE_ITERATIONS = 3
INITIAL_SCALE_RADIUS = 0.4
N_ROTATION_ITERATIONS = 3
INITIAL_ANGLE_STEP = 45
N_TRANSLATION_ITERATIONS = 3
INITIAL_TRANSLATION_RATIO = 0.1

# Outputs / diagnostics
OUTPUT_ROOT = Path("insertion_runs")
SHOW_INTERMEDIATE = True  # inline grids, anchor/bbox visualizations
SHOW_3D = False           # optional interactive trimesh windows
EXPORT_GLB = True


In [7]:
# 04 | Functions — common geometry & diagnostics

def ensure_trimesh(mesh_like):
    if isinstance(mesh_like, trimesh.Scene):
        return mesh_like.dump(concatenate=True)
    if isinstance(mesh_like, trimesh.Trimesh):
        return mesh_like.copy()
    raise TypeError(f"Unsupported mesh type: {type(mesh_like)}")

def normalize(v, eps=1e-8):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    if n < eps:
        return v
    return v / n

def rotation_matrix_from_vectors(a, b):
    """
    Rotation matrix that rotates vector a onto vector b.
    """
    a = normalize(a)
    b = normalize(b)

    cross = np.cross(a, b)
    dot = np.clip(np.dot(a, b), -1.0, 1.0)

    if np.linalg.norm(cross) < 1e-8:
        if dot > 0:
            return np.eye(3)
        # 180-degree rotation around any orthogonal axis
        ortho = np.array([1.0, 0.0, 0.0])
        if abs(a[0]) > 0.9:
            ortho = np.array([0.0, 1.0, 0.0])
        axis = normalize(np.cross(a, ortho))
        return trimesh.transformations.rotation_matrix(np.pi, axis)[:3, :3]

    s = np.linalg.norm(cross)
    kmat = np.array([
        [0, -cross[2], cross[1]],
        [cross[2], 0, -cross[0]],
        [-cross[1], cross[0], 0]
    ])

    R = np.eye(3) + kmat + kmat @ kmat * ((1 - dot) / (s ** 2))
    return R

def mesh_extents_in_frame(mesh, R):
    """
    Extents of vertices after rotation R.
    """
    verts = mesh.vertices @ R.T
    return verts.max(axis=0) - verts.min(axis=0)

def to_o3d_mesh(mesh_like, is_objaverse_model=False):
    """Convert without applying platform removal twice; mirror project's color conversion."""
    if isinstance(mesh_like, o3d.geometry.TriangleMesh):
        return mesh_like
    tmesh = ensure_trimesh(mesh_like)
    if is_objaverse_model:
        tmesh = ensure_trimesh(_remove_platform_only(tmesh))
    result = o3d.geometry.TriangleMesh()
    result.vertices = o3d.utility.Vector3dVector(np.asarray(tmesh.vertices))
    result.triangles = o3d.utility.Vector3iVector(np.asarray(tmesh.faces))
    visual = tmesh.visual.to_color() if hasattr(tmesh.visual, "to_color") else tmesh.visual
    colors = getattr(visual, "vertex_colors", None)
    if colors is not None and len(colors) == len(tmesh.vertices):
        result.vertex_colors = o3d.utility.Vector3dVector(np.asarray(colors)[:, :3] / 255.0)
    result.compute_vertex_normals()
    return result


def show_image(path):
    with Image.open(path) as image:
        display(image.copy())


def log_stage(name):
    print(f"\n{'=' * 72}\n{name}\n{'=' * 72}")


def select_anchor_cluster(
    anchor_points,
    anchor_normals,
    extent,
    radius_ratio,
    min_samples
):
    anchor_points = np.asarray(anchor_points)
    anchor_normals = np.asarray(anchor_normals)

    n_points = len(anchor_points)

    if n_points == 0:
        raise ValueError("No valid anchor points found")

    # Try clustering with progressively relaxed requirements.
    for current_min_samples in range(
        min(min_samples, n_points), 1, -1
    ):
        labels = DBSCAN(
            eps=extent * radius_ratio,
            min_samples=current_min_samples
        ).fit_predict(anchor_points)

        valid_labels = labels[labels != -1]

        if len(valid_labels) == 0:
            continue

        best_label = np.bincount(valid_labels).argmax()

        indices = np.where(labels == best_label)[0]

        cluster_points = anchor_points[indices]
        center = cluster_points.mean(axis=0)

        best_idx = indices[
            np.argmin(
                np.linalg.norm(cluster_points - center, axis=1)
            )
        ]

        print(
            f"Anchor clustering: {len(indices)}/{n_points} points, "
            f"min_samples={current_min_samples}"
        )

        return anchor_points[best_idx], anchor_normals[best_idx]

    # Fallback: select the medoid of all valid points.
    distances = np.linalg.norm(
        anchor_points[:, None, :] - anchor_points[None, :, :],
        axis=-1
    )

    best_idx = np.argmin(distances.sum(axis=1))

    print("WARNING: No consistent cluster found. Using medoid fallback.")

    return anchor_points[best_idx], anchor_normals[best_idx]


# Semantic orientation & view selection

FRONT_VIEW_SELECTION_SYSTEM_PROMPT = """
You are a visual reasoning model for 3D object editing.

The input image contains numbered views of the
same 3D object from different camera positions.

Identify the view that most directly shows
the object's intrinsic front side.

The object may have an arbitrary 3D orientation.
Do not assume that it is standing upright.

Compare all views and identify characteristic
features that indicate the object's front.

Return a JSON object with the following fields:
- reasoning: a brief explanation of your selection.
- front_view: the integer ID of the selected view.
"""

def create_views_grid(rendered_info, cols=6, size=320):
    """
    Create a numbered grid of rendered views. So that model has access to all views at the same time
    """

    rows = math.ceil(len(rendered_info) / cols)

    grid = Image.new(
        "RGB",
        (cols * size, rows * (size + 30)),
        (35, 35, 35)
    )

    draw = ImageDraw.Draw(grid)

    for i, (_, path) in enumerate(rendered_info):
        image = Image.open(path).convert("RGB")
        image = image.resize((size, size))

        x = (i % cols) * size
        y = (i // cols) * (size + 30)

        grid.paste(image, (x, y + 30))
        draw.text(
            (x + 10, y + 8),
            f"View {i + 1:03d}",
            fill="white"
        )

    return grid

def estimate_object_orientation(
    mesh,
    rendered_info,
    grid_path,
    ollama_host,
    model_name="qwen3-vl:8b-instruct"
):
    """Estimate object-centric coordinate axes."""

    client = ollama.Client(host=ollama_host)

    response = client.chat(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": FRONT_VIEW_SELECTION_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": "Identify the most frontal view.",
                "images": [str(grid_path)]
            }
        ],
        format="json",
        options={
            "temperature": 0.05,
            "num_ctx": 16384
        }
    )

    result = json.loads(response["message"]["content"])
    print("VLM response:", result)

    front_idx = int(result["front_view"]) - 1

    # display(Image.open(rendered_info[front_idx][1]))

    if not 0 <= front_idx < len(rendered_info):
        raise ValueError("Invalid front view index")

    # Use the same mesh center as the renderer
    mesh_o3d = to_o3d_mesh(mesh)
    center = mesh_o3d.get_axis_aligned_bounding_box().get_center()

    eye_pos, _ = rendered_info[front_idx]

    # Assume Y-up orientation
    up = np.array([0., 1., 0.])

    # Project front direction onto the horizontal plane
    front = np.asarray(eye_pos) - center
    front -= np.dot(front, up) * up

    if np.linalg.norm(front) < 1e-8:
        raise ValueError("Invalid frontal camera direction")

    front /= np.linalg.norm(front)

    # Compute anatomical directions
    left = np.cross(up, front)
    left /= np.linalg.norm(left)

    return {
        "center": center,
        "front_view_idx": front_idx,
        "front": front,
        "back": -front,
        "left": left,
        "right": -left,
        "up": up,
        "down": -up
    }

def classify_views(rendered_info, orientation, threshold=0.35):
    """
    Classify camera views using object-centric axes. No VLM is used
    Example output:
    {
    "index": 8,
    "eye_pos": array([1.8, 0.9, 1.2]),
    "path": "rendered_views/view_009.png",
    "labels": ["front", "left", "up"],
    "scores": {
        "front": 0.51,
        "back": -0.51,
        "left": 0.77,
        "right": -0.77,
        "up": 0.38,
        "down": -0.38
    }
}
    """

    directions = [
        "front", "back",
        "left", "right",
        "up", "down"
    ]

    center = orientation["center"]
    classified_views = []

    for i, (eye_pos, path) in enumerate(rendered_info):

        camera_dir = np.asarray(eye_pos) - center
        camera_dir /= np.linalg.norm(camera_dir)
        # scores of each direction are decided by dot product with orientation vector
        scores = {
            name: float(np.dot(camera_dir, orientation[name]))
            for name in directions
        }

        labels = [
            name for name in directions
            if scores[name] >= threshold
        ]

        classified_views.append({
            "index": i,
            "eye_pos": eye_pos,
            "path": path,
            "labels": labels,
            "scores": scores
        })

    return classified_views

def select_relevant_views(
    classified_views,
    directions,
    threshold=0.35,
    max_views=6
):
    """
    Select and rank views matching all requested directions.
    Selection is based on scores for each direction and not labels
    """

    relevant_views = []

    for view in classified_views:
        scores = [
            view["scores"][direction]
            for direction in directions
        ]

        # Check all spatial constraints
        if all(score >= threshold for score in scores):

            relevance = np.mean(scores)

            relevant_views.append(
                (view["index"], relevance)
            )

    # Sort views by relevance
    relevant_views.sort(
        key=lambda item: item[1],
        reverse=True
    )

    return [
        index for index, score
        in relevant_views[:max_views]
    ]


# Attachment point grounding

ATTACHMENT_POINT_SYSTEM_PROMPT = """
You are a visual reasoning model for 3D object editing.

Image 1 shows the original object.
The remaining images show the component to attach.

Imagine the component attached to the original object
according to the user's instructions.

Predict the single point on the original object's
surface where the new component should connect.

The point represents the center of the contact area
between the two objects.

Return coordinates relative to Image 1 only.
Use integer coordinates normalized to [0, 1000].

Return a JSON object with the following fields:
- is_found: boolean
- point_2d: [x, y] or null if not found
"""
def generate_attachment_point(
    ollama_host,
    image_path,
    user_prompt,
    rendered_new_object_info,
    model_name="qwen3-vl:8b-instruct"
):
    """Predict a surface attachment point."""
    reference_images = [element[1] for element in rendered_new_object_info]
    with Image.open(image_path) as image:
        width, height = image.size

    client = ollama.Client(host=ollama_host)

    response = client.chat(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": ATTACHMENT_POINT_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt,
                "images": [
                    str(image_path),
                    *[str(path) for path in reference_images]
                ]
            }
        ],
        format="json",
        options={
            "temperature": 0.1,
            "num_ctx": 16384,
            "num_predict": 256
        }
    )

    result = json.loads(response["message"]["content"])

    if not result["is_found"]:
        return None

    x, y = result["point_2d"]
    if not all(np.isfinite(v) and 0 <= v <= 1000 for v in (x, y)):
        raise ValueError(f"Invalid anchor coordinates: {result}")

    return (
        round(x / 1000 * (width - 1)),
        round(y / 1000 * (height - 1))
    )

def pixel_to_3d(
    mesh,
    point_2d,
    eye_pos,
    width=1008,
    height=1008,
    frame_scale=1.8
):
    """
    Project a 2D pixel onto the mesh surface using raycasting.
    """

    # Use the same geometry as in render_n_views
    mesh_o3d = to_o3d_mesh(mesh)

    bbox = mesh_o3d.get_axis_aligned_bounding_box()
    center = bbox.get_center()
    extent = bbox.get_max_extent()

    half_size = extent / 2 * frame_scale
    aspect = width / height

    # Camera coordinate system
    z_cam = eye_pos - center
    z_cam /= np.linalg.norm(z_cam)

    if abs(z_cam[1]) > 0.99:
        up = np.array([0, 0, -1 if z_cam[1] > 0 else 1])
    else:
        up = np.array([0, 1, 0])

    x_cam = np.cross(up, z_cam)
    x_cam /= np.linalg.norm(x_cam)
    y_cam = np.cross(z_cam, x_cam)

    # Convert pixel coordinates to orthographic coordinates
    px, py = point_2d

    x = (2 * (px + 0.5) / width - 1) * half_size * aspect
    y = (1 - 2 * (py + 0.5) / height) * half_size

    # Construct an orthographic ray
    origin = eye_pos + x * x_cam + y * y_cam
    direction = -z_cam

    ray_mesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh_o3d)

    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(ray_mesh)

    ray = np.concatenate([origin, direction])
    rays = o3d.core.Tensor([ray], dtype=o3d.core.Dtype.Float32)

    hits = scene.cast_rays(rays)

    t_hit = hits["t_hit"].numpy()[0]

    if not np.isfinite(t_hit):
        return None, None

    anchor_point = origin + t_hit * direction
    anchor_normal = hits["primitive_normals"].numpy()[0]

    return anchor_point, anchor_normal

def validate_anchor_point(
    point,
    orientation,
    directions
):
    """
    Check whether a point satisfies spatial constraints.
    """

    relative_point = point - orientation["center"]

    return all(
        np.dot(relative_point, orientation[d]) >= 0
        for d in directions
    )


# Anchor rendering

def _auto_camera_up(center, eye):
    """
    Place camera to the up
    :param center:
    :param eye:
    :return:
    """
    eye = np.asarray(eye, dtype=float)
    center = np.asarray(center, dtype=float)

    z_cam = eye - center
    z_cam /= np.linalg.norm(z_cam)

    if abs(z_cam[1]) > 0.99:
        return np.array([0.0, 0.0, -1.0]) if z_cam[1] > 0 else np.array([0.0, 0.0, 1.0])
    return np.array([0.0, 1.0, 0.0])

def rendered_info_to_views(mesh_input, rendered_info):
    """
    Converts rendered info to {
        "view_001": (eye_pos, up),
        "view_002": (eye_pos, up),
        ...
    }
    :param mesh_input:
    :param rendered_info:
    :return:
    """
    if isinstance(mesh_input, trimesh.Scene):
        tmesh = mesh_input.dump(concatenate=True)
        mesh_o3d = to_o3d_mesh(tmesh, is_objaverse_model=False)
    elif isinstance(mesh_input, trimesh.Trimesh):
        mesh_o3d = to_o3d_mesh(mesh_input, is_objaverse_model=False)
    elif isinstance(mesh_input, str):
        mesh_o3d = o3d.io.read_triangle_mesh(mesh_input)
    else:
        mesh_o3d = mesh_input

    center = mesh_o3d.get_axis_aligned_bounding_box().get_center()

    views = {}
    for idx, (eye, _) in enumerate(rendered_info):
        eye = np.asarray(eye, dtype=float)
        up = _auto_camera_up(center, eye)
        views[f"view_{idx + 1:03d}"] = (eye, up)

    return views

def rerender_scene_with_anchor(
    mesh,
    anchor_point,
    rendered_info,
    output_dir="rendered_anchor_views",
    width=1008,
    height=1008,
    frame_scale=1.8,
    marker_radius_ratio=0.025,
):
    """Render the original mesh with a red attachment marker."""

    # Convert the original mesh while preserving colors
    base_mesh = to_o3d_mesh(mesh)

    bbox = base_mesh.get_axis_aligned_bounding_box()
    extent = bbox.get_max_extent()

    # Create a red anchor marker
    marker = o3d.geometry.TriangleMesh.create_sphere(
        radius=extent * marker_radius_ratio
    )

    marker.translate(anchor_point)
    marker.paint_uniform_color([1.0, 0.0, 0.0])
    marker.compute_vertex_normals()

    # Combine geometries after color conversion
    scene = base_mesh + marker

    # Restore original camera positions
    views = rendered_info_to_views(base_mesh, rendered_info)

    return render_n_views(
        scene,
        output_dir=output_dir,
        width=width,
        height=height,
        views=views,
        frame_scale=frame_scale,
        is_objaverse_model=False
    )


# Bounding box prediction (anchor coordinates + visibility)

INSERTION_BOUNDING_BOX_SYSTEM_PROMPT = """
Image 1 shows a 3D object with a red attachment marker.
The remaining images show the component to insert.

Imagine the new component attached to the object
at the red marker.

Predict the bounding box of the entire imagined
component, including its occluded portions.

The bounding box may overlap the original object
and extend into the background.

Estimate a suitable size and orientation.

Return coordinates relative to Image 1 only,
using integer coordinates normalized to [0, 1000].

Return only JSON with the following fields:
- is_found: boolean
- bbox_2d: [xmin, ymin, xmax, ymax]

Return is_found=false only if the placement
cannot be estimated at all.
"""

RELATIVE_INSERTION_BBOX_SYSTEM_PROMPT = """
You are a spatial reasoning model for 3D object insertion.

Image 1 shows the original object with an attachment marker.
The remaining images show different views of the component
to be inserted.

Imagine the component physically attached at the given anchor.

The reference images illustrate the component's shape
and appearance, NOT its intended scale.
The component was rendered independently.
Do not transfer its apparent size from the reference images.

Consider:
- The geometry and proportions of the inserted component.
- Its natural size relative to the original object.
- Its orientation after attachment.
- The current camera viewing angle.
- The user request.

Predict the tightest bounding box of the entire imagined
component, including its occluded portions.

The bounding box must contain the attachment anchor,
but the anchor is not necessarily its center.

Do not assume a square bounding box.
Account for the current viewing angle.
Do not surround the original object.

Return absolute image coordinates as integers
normalized to [0, 1000].

Return only JSON:
{
    "xmin": integer,
    "ymin": integer,
    "xmax": integer,
    "ymax": integer
}
"""

def generate_relative_insertion_bbox(
    ollama_host,
    image_path,
    reference_images,
    anchor_2d,
    user_prompt,
    model_name="qwen3-vl:8b-instruct"
):
    with Image.open(image_path) as image:
        width, height = image.size

    ax, ay = anchor_2d

    nx = round(ax / width * 1000)
    ny = round(ay / height * 1000)

    content = f"""
    User request: {user_prompt}

    Attachment anchor in Image 1:
    x = {nx}
    y = {ny}

    Predict the absolute bounding box coordinates
    of the imagined inserted component.
    """

    client = ollama.Client(host=ollama_host)

    response = client.chat(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": RELATIVE_INSERTION_BBOX_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": content,
                "images": [
                    str(image_path),
                    *[str(p) for p in reference_images]
                ]
            }
        ],
        format="json",
        think=False,
        options={
            "temperature": 0.1,
            "num_ctx": 16384
        }
    )

    result = json.loads(response["message"]["content"])

    print("Raw VLM response:", result)

    coords = [float(result[k]) for k in ("xmin", "ymin", "xmax", "ymax")]
    if not all(np.isfinite(v) and 0 <= v <= 1000 for v in coords):
        return {"is_found": False, "bbox_2d": None, "anchor_inside": False}

    x1 = coords[0] * width / 1000
    y1 = coords[1] * height / 1000
    x2 = coords[2] * width / 1000
    y2 = coords[3] * height / 1000

    bbox = [round(v) for v in [x1, y1, x2, y2]]

    anchor_inside = (
        x1 <= ax <= x2 and
        y1 <= ay <= y2
    )

    print("Anchor inside:", anchor_inside)

    return {
        "is_found": x1 < x2 and y1 < y2,
        "bbox_2d": bbox,
        "anchor_inside": anchor_inside
    }

def visualize_bbox_result(image_path, bbox_result):
    """
    Visualizes bounding box result
    :param image_path:
    :param bbox_result:
    :return:
    """
    image = np.array(Image.open(image_path).convert("RGB"))

    plt.figure(figsize=(8, 8))
    plt.imshow(image)

    if bbox_result.get("is_found") and bbox_result.get("bbox_2d") is not None:
        x1, y1, x2, y2 = bbox_result["bbox_2d"]
        plt.gca().add_patch(
            plt.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                edgecolor="yellow",
                facecolor="none",
                linewidth=2
            )
        )

    plt.axis("off")
    plt.show()

def is_anchor_visible(image_path, anchor_2d, radius=30):
    image = np.array(
        Image.open(image_path).convert("RGB"),
        dtype=np.float32
    )

    height, width = image.shape[:2]
    x, y = np.round(anchor_2d).astype(int)

    if not (0 <= x < width and 0 <= y < height):
        return False

    region = image[
        max(0, y-radius):min(height, y+radius+1),
        max(0, x-radius):min(width, x+radius+1)
    ]

    red = (
        (region[:, :, 0] > 100) &
        (region[:, :, 0] > region[:, :, 1] * 1.5) &
        (region[:, :, 0] > region[:, :, 2] * 1.5)
    )

    return np.sum(red) > 10


# Voxel voting & fitting initialization

def create_voxel_grid(mesh, resolution=64, padding=0.3):
    """Create a regular voxel grid around the original mesh."""

    extent = mesh.extents.max()
    center = mesh.bounds.mean(axis=0)

    grid_size = extent * (1 + 2 * padding)
    voxel_size = grid_size / resolution

    # Generate voxel center coordinates
    axis = (
        -grid_size / 2
        + (np.arange(resolution) + 0.5) * voxel_size
    )

    x, y, z = np.meshgrid(axis, axis, axis, indexing="ij")

    voxel_points = np.stack(
        [x, y, z], axis=-1
    ).reshape(-1, 3)

    voxel_points += center

    return voxel_points, voxel_size

def project_points_to_image(
    points,
    eye_pos,
    center,
    extent,
    width=1008,
    height=1008,
    frame_scale=1.8,
    up=None
):
    """
    Project 3D points onto an orthographic image.
    """

    half_size = extent / 2 * frame_scale
    aspect = width / height

    # Camera coordinate system
    z_cam = np.asarray(eye_pos) - center
    z_cam /= np.linalg.norm(z_cam)

    if up is None:
        if abs(z_cam[1]) > 0.99:
            up = np.array([
                0., 0., -1. if z_cam[1] > 0 else 1.
            ])
        else:
            up = np.array([0., 1., 0.])

    x_cam = np.cross(up, z_cam)
    x_cam /= np.linalg.norm(x_cam)

    y_cam = np.cross(z_cam, x_cam)

    # Project points onto camera axes
    relative_points = points - center

    x_proj = relative_points @ x_cam
    y_proj = relative_points @ y_cam

    # Convert to pixel coordinates
    px = (
        (x_proj / (half_size * aspect) + 1)
        * width / 2 - 0.5
    )

    py = (
        (1 - y_proj / half_size)
        * height / 2 - 0.5
    )

    return np.column_stack([px, py])

def vote_for_voxels(
    voxel_points,
    bbox_results,
    center,
    extent,
    width=1008,
    height=1008,
    frame_scale=1.8
):
    """Calculate multi-view support for each voxel."""

    votes = np.zeros(len(voxel_points), dtype=np.int32)

    valid_results = [
        result for result in bbox_results
        if result["is_found"]
        and result["bbox_2d"] is not None
    ]

    if not valid_results:
        raise ValueError("No valid bounding boxes")

    for result in valid_results:

        projected = project_points_to_image(
            points=voxel_points,
            eye_pos=result["eye_pos"],
            center=center,
            extent=extent,
            width=width,
            height=height,
            frame_scale=frame_scale,
            up=result.get("up")
        )

        px = projected[:, 0]
        py = projected[:, 1]

        x1, y1, x2, y2 = result["bbox_2d"]

        # Check whether projected voxels are inside the box
        inside = (
            (px >= x1) &
            (px <= x2) &
            (py >= y1) &
            (py <= y2)
        )

        votes += inside.astype(np.int32)

    scores = votes / len(valid_results)

    return scores

def visualize_voxel_grid(mesh, voxel_points, anchor_point):
    """Visualize selected voxel centers and the attachment point."""

    cloud = trimesh.points.PointCloud(
        vertices=voxel_points,
        colors=[255, 0, 0, 255]
    )

    marker = trimesh.creation.icosphere(
        radius=mesh.extents.max() * 0.025
    )

    marker.apply_translation(anchor_point)
    marker.visual.face_colors = [255, 0, 0, 255]

    scene = trimesh.Scene([mesh, cloud, marker])
    return scene

def select_fitting_views(
    bbox_results,
    selected_views,
    n_views=3
):
    """Select relevant views for mesh fitting."""

    valid_views = {
        r["view_idx"]: r
        for r in bbox_results
        if r["is_found"]
    }

    fitting_views = [
        valid_views[i]
        for i in selected_views
        if i in valid_views
    ]

    return fitting_views[:n_views]

def initialize_scale_transform(
    base_mesh,
    new_object_mesh,
    selected_voxels,
    anchor_point,
    anchor_normal,
    penetration_ratio=0.08
):
    """
    Returns:
        init_scale,
        base_rotation (3x3),
        base_translation (3,),
        attachment_axis_local (3,)
    """
    base_mesh = ensure_trimesh(base_mesh)
    obj_mesh = ensure_trimesh(new_object_mesh)

    voxel_min = selected_voxels.min(axis=0)
    voxel_max = selected_voxels.max(axis=0)
    voxel_extent = voxel_max - voxel_min
    voxel_center = selected_voxels.mean(axis=0)

    # Use axis-aligned bounding box extents
    obj_extent = obj_mesh.extents
    attachment_axis_idx = np.argmin(obj_extent)   # thinnest axis
    attachment_axis_local = np.eye(3)[attachment_axis_idx]

    # Align thinnest object axis with anchor normal
    normal = normalize(anchor_normal)
    base_rotation = rotation_matrix_from_vectors(
        attachment_axis_local,
        normal
    )

    rotated_extent = mesh_extents_in_frame(obj_mesh, base_rotation)

    # Use voxel support size as initial scale estimate
    # robust scalar: mean of non-small extents
    target_size = np.mean(np.sort(voxel_extent)[-2:])
    source_size = np.mean(np.sort(rotated_extent)[-2:])
    init_scale = float(target_size / max(source_size, 1e-8))

    # Build transformed vertices only for translation estimation
    verts = obj_mesh.vertices @ base_rotation.T
    verts = verts * init_scale

    # Tangential alignment: object center -> voxel cloud center
    obj_center = verts.mean(axis=0)

    normal = normalize(anchor_normal)

    # Shift along normal so object slightly penetrates the mesh
    proj = verts @ normal
    min_proj = proj.min()

    anchor_proj = np.dot(anchor_point, normal)
    penetration = base_mesh.extents.max() * penetration_ratio

    delta_n = (anchor_proj - penetration) - min_proj

    # Tangent-plane alignment using voxel center
    obj_center_tangent = obj_center - np.dot(obj_center, normal) * normal
    voxel_center_tangent = voxel_center - np.dot(voxel_center, normal) * normal
    delta_tangent = voxel_center_tangent - obj_center_tangent

    base_translation = delta_tangent + delta_n * normal

    return init_scale, base_rotation, base_translation, attachment_axis_local

def transform_object_mesh(
    new_object_mesh,
    scale,
    rotation_matrix,
    translation
):
    obj_mesh = ensure_trimesh(new_object_mesh).copy()

    verts = obj_mesh.vertices @ rotation_matrix.T
    verts = verts * scale
    verts = verts + translation

    obj_mesh.vertices = verts
    return obj_mesh


# Candidate rendering

def render_scene_views_with_object(
    base_mesh,
    object_mesh,
    fitting_views,
    output_dir,
    image_size=512,
    frame_scale=1.8
):
    os.makedirs(output_dir, exist_ok=True)

    base_o3d = to_o3d_mesh(ensure_trimesh(base_mesh), is_objaverse_model=False)
    obj_o3d = to_o3d_mesh(ensure_trimesh(object_mesh), is_objaverse_model=False)

    bbox = base_o3d.get_axis_aligned_bounding_box()
    center = bbox.get_center()
    extent = bbox.get_max_extent()

    half_size = (extent / 2.0) * frame_scale
    aspect = 1.0

    renderer = o3d.visualization.rendering.OffscreenRenderer(image_size, image_size)

    material_base = o3d.visualization.rendering.MaterialRecord()
    material_base.shader = "defaultLit"

    material_obj = o3d.visualization.rendering.MaterialRecord()
    material_obj.shader = "defaultLit"

    renderer.scene.add_geometry("base_mesh", base_o3d, material_base)
    renderer.scene.add_geometry("inserted_mesh", obj_o3d, material_obj)
    renderer.scene.set_background(np.array([0.2, 0.2, 0.2, 1.0]))

    rendered_paths = []

    for local_idx, view in enumerate(fitting_views):
        eye = np.asarray(view["eye_pos"], dtype=float)
        up = np.asarray(view["up"], dtype=float)

        renderer.scene.camera.look_at(center, eye, up)
        renderer.scene.camera.set_projection(
            o3d.visualization.rendering.Camera.Projection.Ortho,
            -half_size * aspect, half_size * aspect,
            -half_size, half_size,
            0.1, extent * 10.0
        )

        img = renderer.render_to_image()
        path = os.path.join(output_dir, f"view_{local_idx:02d}.png")
        o3d.io.write_image(path, img)
        rendered_paths.append(path)

    del renderer
    return rendered_paths


def build_candidate_collage(image_paths, out_path):
    images = [Image.open(p).convert("RGB") for p in image_paths]

    w, h = images[0].size
    header_h = 40
    canvas = Image.new("RGB", (w * len(images), h + header_h), "white")
    draw = ImageDraw.Draw(canvas)

    for i, img in enumerate(images):
        canvas.paste(img, (i * w, header_h))
        draw.text((i * w + 10, 10), f"View {i+1}", fill="black")

    canvas.save(out_path)
    return out_path

def build_candidates_grid(collage_paths, out_path, columns=2):
    collages = [Image.open(p).convert("RGB") for p in collage_paths]

    w, h = collages[0].size
    rows = math.ceil(len(collages) / columns)

    canvas = Image.new("RGB", (columns * w, rows * (h + 24)), "white")
    draw = ImageDraw.Draw(canvas)

    for idx, img in enumerate(collages):
        x = (idx % columns) * w
        y = (idx // columns) * (h + 24)
        draw.text((x + 10, y + 5), f"Candidate {idx + 1}", fill="black")
        canvas.paste(img, (x, y + 24))

    canvas.save(out_path)
    return out_path

def generate_scale_candidates(center_scale, radius=0.4, n=5):
    """
    Example:
    center=1.0, radius=0.4, n=5
    -> [0.6, 0.8, 1.0, 1.2, 1.4]
    """
    offsets = np.linspace(-radius, radius, n)
    candidates = [center_scale * (1.0 + o) for o in offsets]
    return [float(max(1e-4, c)) for c in candidates]

def generate_rotation_candidates(
    current_rotation,
    current_translation,
    anchor_point,
    axis,
    angle_step
):
    candidates = []

    for angle in [-angle_step, 0, angle_step]:

        delta_rotation = trimesh.transformations.rotation_matrix(
            np.deg2rad(angle),
            axis
        )[:3, :3]

        rotation = delta_rotation @ current_rotation

        translation = (
            anchor_point
            + delta_rotation @ (current_translation - anchor_point)
        )

        candidates.append({
            "angle": angle,
            "rotation": rotation,
            "translation": translation
        })

    random.shuffle(candidates)

    return candidates

def build_anchor_frame(anchor_normal, object_front=None):
    """
    Returns local orthonormal basis:
    tangent_1, tangent_2, normal

    tangent_1: preferred lateral direction on surface
    tangent_2: preferred vertical-like direction on surface
    normal: outward normal
    """
    n = normalize(anchor_normal)

    if object_front is None:
        object_front = np.array([1.0, 0.0, 0.0])

    f = normalize(object_front)

    # remove normal component from front
    t1 = f - np.dot(f, n) * n
    if np.linalg.norm(t1) < 1e-8:
        # fallback
        up = np.array([0.0, 1.0, 0.0])
        t1 = np.cross(up, n)
        if np.linalg.norm(t1) < 1e-8:
            up = np.array([1.0, 0.0, 0.0])
            t1 = np.cross(up, n)

    t1 = normalize(t1)
    t2 = normalize(np.cross(n, t1))

    return t1, t2, n

def generate_translation_candidates(
    base_translation,
    axis,
    step,
    include_center=True
):
    candidates = []

    if include_center:
        candidates.append({
            "label": "center",
            "translation": base_translation.copy(),
            "delta": 0.0
        })

    candidates.append({
        "label": "minus",
        "translation": base_translation - axis * step,
        "delta": -step
    })

    candidates.append({
        "label": "plus",
        "translation": base_translation + axis * step,
        "delta": step
    })

    return candidates


# VLM candidate selection / ranking

TRANSFORMATION_PROMPTS = {
    "scale": """
You are evaluating different scales of a component
inserted into a 3D object.

Each input image represents one candidate.
Each image contains different camera views
of the same candidate.

Candidate numbering follows the input image order.

Compare the same camera viewpoints across candidates.

Evaluate:
- The size of the inserted component relative to the base object.
- Natural proportions.
- Consistency with the user request.

Do not prefer larger or more visually prominent components.
Do not judge image quality.
Focus on the relative size of the inserted component.
""",
    "rotation": """
You are evaluating different orientations of a component
inserted into a 3D object.

Each input image represents one candidate.
Each image contains different camera viewpoints.

Compare the same camera viewpoints across candidates.

Evaluate:
- The orientation of the inserted component
  relative to the base object.
- Natural alignment.
- Consistency with the user request.

Do not prefer larger or more visually prominent components.
Do not judge image quality.
Focus on the relative orientation of the inserted component.
""",
    "translation": """
You are a visual evaluator of 3D object placement.

Each input image represents one candidate and contains multiple views of that candidate.
Each candidate shows the same inserted object attached to the same base object.

Scale, orientation and object identity are fixed.
Only the position of the inserted object differs.

Your task is to choose which candidate has the best placement.

Evaluation criteria:
- the inserted object should be attached at the intended anchor area
- it should look naturally connected to the base object
- it should not float too far away
- it should not intersect the base object too deeply
- it should not be obviously misplaced
- prefer candidates whose placement is physically plausible

Do not prefer a candidate only because the inserted object is larger
or more visible.
""",
}

def select_best_transformation_candidate(candidate_collage_paths, user_prompt, transformation,
                                         ollama_host, model_name="qwen3-vl:8b-instruct",
                                         mode="selection"):
    """Return a validated 1-based candidate ID (and optional full ranking)."""
    n = len(candidate_collage_paths)
    if n < 2:
        raise ValueError("At least two candidates are required")
    system_prompt = TRANSFORMATION_PROMPTS[transformation] + """

Each input image is one candidate, containing views of the same scene.
Compare corresponding views across candidates.
Candidate IDs follow the order of the images, starting from 1.
"""
    if mode == "selection":
        system_prompt += '\nSelect the best candidate. Return JSON only: {"best_candidate": integer, "reason": string}.'
    elif mode == "ranking":
        system_prompt += '\nRank all candidates best to worst; use each ID exactly once. Return JSON only: {"ranking": [integer, ...], "reason": string}.'
    else:
        raise ValueError(f"Unknown selection mode: {mode}")
    client = ollama.Client(host=ollama_host)
    response = client.chat(
        model=model_name,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": f"User request:\n{user_prompt}",
                   "images": [str(p) for p in candidate_collage_paths]}],
        format="json", think=False,
        options={"temperature": 0.1, "num_ctx": 16384, "num_predict": 512}
    )
    result = json.loads(response["message"]["content"])
    if mode == "ranking":
        ranking = result.get("ranking")
        if not isinstance(ranking, list) or any(type(i) is not int for i in ranking) or sorted(ranking) != list(range(1, n + 1)):
            raise ValueError(f"Invalid ranking: {result}")
        result["best_candidate"] = ranking[0]
    chosen = result.get("best_candidate")
    if type(chosen) is not int or not 1 <= chosen <= n:
        raise ValueError(f"Invalid selection: {result}")
    print(f"VLM {transformation}: {result}")
    return result


# Scale search

def run_scale_search(
    base_mesh,
    new_object_mesh,
    fitting_views,
    base_rotation,
    base_translation,
    init_scale,
    user_prompt,
    anchor_point,
    output_root="scale_search",
    n_iterations=3,
    n_candidates=5,
    initial_radius=0.4,
    image_size=512,
    *, ollama_host, model_name, mode="selection", frame_scale=1.8, show_intermediate=True
):
    os.makedirs(output_root, exist_ok=True)

    if not np.isfinite(init_scale) or init_scale <= 0:
        raise ValueError("Initial scale must be positive")
    current_scale = init_scale
    current_radius = initial_radius

    history = []

    for iteration in range(n_iterations):
        print(f"\n=== Scale iteration {iteration + 1} ===")
        print(f"Center scale: {current_scale:.6f}")
        print(f"Search radius: {current_radius:.6f}")

        iter_dir = os.path.join(output_root, f"iter_{iteration + 1:02d}")
        os.makedirs(iter_dir, exist_ok=True)

        candidate_scales = generate_scale_candidates(
            center_scale=current_scale,
            radius=current_radius,
            n=n_candidates
        )
        random.shuffle(candidate_scales)

        candidate_infos = []
        collage_paths = []

        for cand_idx, scale_value in enumerate(candidate_scales, start=1):
            #candidate_name = f"Candidate {cand_idx} | scale={scale_value:.4f}"

            candidate_translation = (
                anchor_point
                + (scale_value / init_scale)
                * (base_translation - anchor_point)
            )

            transformed_mesh = transform_object_mesh(
                new_object_mesh=new_object_mesh,
                scale=scale_value,
                rotation_matrix=base_rotation,
                translation=candidate_translation
            )

            render_dir = os.path.join(iter_dir, f"candidate_{cand_idx:02d}_views")
            view_paths = render_scene_views_with_object(
                base_mesh=base_mesh,
                object_mesh=transformed_mesh,
                fitting_views=fitting_views,
                output_dir=render_dir,
                image_size=image_size,
                frame_scale=frame_scale
            )

            collage_path = os.path.join(iter_dir, f"candidate_{cand_idx:02d}_collage.png")
            build_candidate_collage(
                image_paths=view_paths,
                #candidate_name=candidate_name,
                out_path=collage_path
            )

            collage_paths.append(collage_path)

            candidate_infos.append({
                "candidate_idx": cand_idx,
                "scale": scale_value,
                "rotation": base_rotation.copy(),
                "translation": candidate_translation.copy(),
                "mesh": transformed_mesh,
                "view_paths": view_paths,
                "collage_path": collage_path
            })

        # Grid for you
        grid_path = os.path.join(iter_dir, "all_candidates_grid.png")
        build_candidates_grid(
            collage_paths=collage_paths,
            out_path=grid_path,
            columns=1
        )

        print(f"Saved grid: {grid_path}")
        if show_intermediate:
            show_image(grid_path)

        # Send the separate collages to VLM
        vlm_result = select_best_transformation_candidate(
            candidate_collage_paths=collage_paths,
            user_prompt=user_prompt, transformation="scale",
            ollama_host=ollama_host, model_name=model_name, mode=mode
        )

        best_idx = int(vlm_result["best_candidate"]) - 1
        if not 0 <= best_idx < len(candidate_infos):
            raise ValueError("Invalid candidate ID")
        best_info = candidate_infos[best_idx]

        print(f"Selected candidate: {best_info['candidate_idx']}")
        print(f"Selected scale: {best_info['scale']:.6f}")

        history.append({
            "iteration": iteration + 1,
            "grid_path": grid_path,
            "vlm_result": vlm_result,
            "candidate_infos": candidate_infos,
            "selected_candidate": best_info
        })

        current_scale = best_info["scale"]
        current_radius *= 0.5

    return history


# Rotation search

def run_rotation_search(
    base_mesh,
    new_object_mesh,
    fitting_views,
    anchor_point,
    best_scale,
    base_rotation,
    base_translation,
    user_prompt,
    output_root="rotation_search",
    n_iterations=3,
    initial_angle_step=45,
    image_size=512,
    *, ollama_host, model_name, mode="selection", frame_scale=1.8, show_intermediate=True
):
    current_rotation = base_rotation.copy()
    current_translation = base_translation.copy()
    angle_step = initial_angle_step

    axes = {
        "X": np.array([1., 0., 0.]),
        "Y": np.array([0., 1., 0.]),
        "Z": np.array([0., 0., 1.])
    }

    history = []

    for iteration in range(n_iterations):

        for axis_name, axis in axes.items():

            print(f"\n=== Rotation iteration {iteration+1} ===")
            print(f"Axis: {axis_name}")
            print(f"Angle step: {angle_step:.2f}")

            iter_dir = os.path.join(
                output_root,
                f"iter_{iteration+1:02d}_{axis_name}"
            )

            os.makedirs(iter_dir, exist_ok=True)

            candidates = generate_rotation_candidates(
                current_rotation=current_rotation,
                current_translation=current_translation,
                anchor_point=anchor_point,
                axis=axis,
                angle_step=angle_step
            )

            collage_paths = []

            for i, candidate in enumerate(candidates, start=1):

                transformed_mesh = transform_object_mesh(
                    new_object_mesh=new_object_mesh,
                    scale=best_scale,
                    rotation_matrix=candidate["rotation"],
                    translation=candidate["translation"]
                )

                view_paths = render_scene_views_with_object(
                    base_mesh=base_mesh,
                    object_mesh=transformed_mesh,
                    fitting_views=fitting_views,
                    output_dir=os.path.join(
                        iter_dir, f"candidate_{i:02d}"
                    ),
                    image_size=image_size,
                    frame_scale=frame_scale
                )

                collage_path = os.path.join(
                    iter_dir,
                    f"candidate_{i:02d}.png"
                )

                build_candidate_collage(
                    image_paths=view_paths,
                    out_path=collage_path
                )

                collage_paths.append(collage_path)

                candidate["mesh"] = transformed_mesh
                candidate["scale"] = best_scale
                candidate["candidate_idx"] = i
                candidate["collage_path"] = collage_path
                candidate["view_paths"] = view_paths

            grid_path = os.path.join(
                iter_dir, "all_candidates_grid.png"
            )

            build_candidates_grid(
                collage_paths=collage_paths,
                out_path=grid_path,
                columns=1
            )

            if show_intermediate:
                show_image(grid_path)

            result = select_best_transformation_candidate(
                candidate_collage_paths=collage_paths,
                user_prompt=user_prompt, transformation="rotation",
                ollama_host=ollama_host, model_name=model_name, mode=mode
            )

            best_idx = int(result["best_candidate"]) - 1

            if not 0 <= best_idx < len(candidates):
                raise ValueError("Invalid candidate ID")

            best_candidate = candidates[best_idx]

            current_rotation = best_candidate["rotation"]
            current_translation = best_candidate["translation"]

            print("Selected candidate:", best_idx + 1)
            print("Selected angle:", best_candidate["angle"])

            history.append({
                "iteration": iteration + 1,
                "axis": axis_name,
                "angle_step": angle_step,
                "selected_candidate": best_candidate,
                "candidate_infos": candidates,
                "vlm_result": result,
                "grid_path": grid_path
            })

        angle_step *= 0.5

    return history


# Translation search

def run_translation_search(
    base_mesh,
    new_object_mesh,
    fitting_views,
    anchor_point,
    anchor_normal,
    best_scale,
    base_rotation,
    base_translation,
    object_front,
    user_prompt,
    output_root="translation_search",
    n_iterations=3,
    initial_step=0.08,
    image_size=512,
    *, ollama_host, model_name, mode="selection", frame_scale=1.8, show_intermediate=True
):
    t1, t2, n = build_anchor_frame(
        anchor_normal, object_front
    )

    axes = {
        "tangent_1": t1,
        "tangent_2": t2,
        "normal": n
    }

    current_translation = base_translation.copy()
    step = initial_step
    history = []

    for iteration in range(n_iterations):
        for axis_name, axis in axes.items():

            print(f"\n=== Translation iteration {iteration + 1} ===")
            print(f"Axis: {axis_name}")
            print(f"Step: {step:.4f}")

            iter_dir = os.path.join(
                output_root,
                f"iter_{iteration+1:02d}_{axis_name}"
            )

            os.makedirs(iter_dir, exist_ok=True)

            candidates = generate_translation_candidates(
                base_translation=current_translation,
                axis=axis,
                step=step
            )

            random.shuffle(candidates)

            collage_paths = []

            for i, candidate in enumerate(candidates, start=1):

                transformed_mesh = transform_object_mesh(
                    new_object_mesh=new_object_mesh,
                    scale=best_scale,
                    rotation_matrix=base_rotation,
                    translation=candidate["translation"]
                )

                view_paths = render_scene_views_with_object(
                    base_mesh=base_mesh,
                    object_mesh=transformed_mesh,
                    fitting_views=fitting_views,
                    output_dir=os.path.join(
                        iter_dir, f"candidate_{i:02d}"
                    ),
                    image_size=image_size,
                    frame_scale=frame_scale
                )

                collage_path = os.path.join(
                    iter_dir, f"candidate_{i:02d}.png"
                )

                build_candidate_collage(
                    image_paths=view_paths,
                    out_path=collage_path
                )

                collage_paths.append(collage_path)

                candidate["mesh"] = transformed_mesh
                candidate["scale"] = best_scale
                candidate["candidate_idx"] = i
                candidate["collage_path"] = collage_path
                candidate["view_paths"] = view_paths
                candidate["rotation"] = base_rotation.copy()

            grid_path = os.path.join(
                iter_dir, "all_candidates_grid.png"
            )

            build_candidates_grid(
                collage_paths=collage_paths,
                out_path=grid_path,
                columns=1
            )

            if show_intermediate:
                show_image(grid_path)

            result = select_best_transformation_candidate(
                candidate_collage_paths=collage_paths,
                user_prompt=user_prompt, transformation="translation",
                ollama_host=ollama_host, model_name=model_name, mode=mode
            )

            best_idx = int(result["best_candidate"]) - 1

            if not 0 <= best_idx < len(candidates):
                raise ValueError("Invalid candidate ID")

            best_candidate = candidates[best_idx]

            current_translation = best_candidate["translation"]

            print("VLM selection:", result)
            print("Selected delta:", best_candidate["delta"])

            history.append({
                "iteration": iteration + 1,
                "axis": axis_name,
                "step": step,
                "selected_candidate": best_candidate,
                "candidate_infos": candidates,
                "vlm_result": result,
                "grid_path": grid_path
            })

        step *= 0.5

    return history


# Optional legacy marker-only bbox method (not used by main pipeline)

def _parse_qwen_bbox_response(content, width, height):
    """
    Parses VLM response to collect bounding box coordinates
    :param content:
    :param width:
    :param height:
    :return:
    """
    try:
        result = json.loads(content)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON: {content[:500]!r}") from exc

    if not isinstance(result, dict):
        raise ValueError(f"Invalid response type: {type(result)}")

    if type(result.get("is_found")) is not bool:
        raise ValueError(f"Missing/invalid is_found: {result}")

    if result["is_found"] is False:
        return {"is_found": False, "bbox_2d": None}

    bbox = result.get("bbox_2d")
    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        raise ValueError(f"Invalid bbox_2d: {bbox}")

    coords = [float(v) for v in bbox]

    if not all(0 <= v <= 1000 for v in coords):
        raise ValueError(f"Coordinates outside [0,1000]: {bbox}")

    x1, y1, x2, y2 = coords
    x1 = round(x1 * width / 1000)
    y1 = round(y1 * height / 1000)
    x2 = round(x2 * width / 1000)
    y2 = round(y2 * height / 1000)

    x1 = max(0, min(width - 1, x1))
    y1 = max(0, min(height - 1, y1))
    x2 = max(0, min(width - 1, x2))
    y2 = max(0, min(height - 1, y2))

    if x1 >= x2 or y1 >= y2:
        raise ValueError(f"Collapsed/inverted bbox: {[x1, y1, x2, y2]}")

    return {"is_found": True, "bbox_2d": [x1, y1, x2, y2]}

def generate_insertion_bounding_box_for_image(
    ollama_host,
    image_path,
    inserted_object_image_paths,
    user_prompt,
    model_name="qwen3-vl:8b-instruct",
    temperature=0.1,
    num_ctx=16384,
    max_reference_views=3,
):
    """
    Generates bounding box for one view
    :param ollama_host: ip of server with ollama
    :param image_path: path to view
    :param inserted_object_image_paths: paths to object that should be added
    :param user_prompt: prompt given to the tool
    :param model_name: name of the VLM model
    :param temperature:
    :param num_ctx:
    :param max_reference_views:
    :return:
    """
    with Image.open(image_path) as image:
        width, height = image.size

    reference_paths = [str(p) for p in inserted_object_image_paths[:max_reference_views]]
    images = [str(image_path)] + reference_paths

    content = f"""
    Image 1 shows the target object with a red anchor marker.
    The new component must be attached at the red marker.

    User request:
    {user_prompt}

    Return a bounding box for the imagined inserted component only in Image 1.
    If this view is not suitable or the placement cannot be estimated reliably, return is_found=false.
    """

    client = ollama.Client(host=ollama_host)

    response = client.chat(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": INSERTION_BOUNDING_BOX_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": content,
                "images": images
            }
        ],
        format="json",
        think=False,
        options={
            "temperature": temperature,
            "num_ctx": num_ctx
        }
    )

    parsed = _parse_qwen_bbox_response(
        response["message"]["content"],
        width=width,
        height=height
    )

    return json.dumps(parsed)

def generate_insertion_bounding_boxes(
    ollama_host,
    rendered_anchor_info,
    inserted_object_image_paths,
    user_prompt,
    model_name="qwen3-vl:8b-instruct",
    temperature=0.1,
    num_ctx=16384,
    max_reference_views=3,
):
    """
    Generates bounding box for each view
    :param ollama_host:
    :param rendered_anchor_info:
    :param inserted_object_image_paths:
    :param user_prompt:
    :param model_name:
    :param temperature:
    :param num_ctx:
    :param max_reference_views:
    :return:
    """
    results = []

    for view_idx, (eye_pos, image_path) in enumerate(rendered_anchor_info):
        try:
            raw = generate_insertion_bounding_box_for_image(
                ollama_host=ollama_host,
                image_path=image_path,
                inserted_object_image_paths=inserted_object_image_paths,
                user_prompt=user_prompt,
                model_name=model_name,
                temperature=temperature,
                num_ctx=num_ctx,
                max_reference_views=max_reference_views,
            )
            parsed = json.loads(raw)
        except Exception as exc:
            parsed = {
                "is_found": False,
                "bbox_2d": None,
                "error": str(exc),
            }

        parsed["view_idx"] = view_idx
        parsed["eye_pos"] = np.asarray(eye_pos).tolist()
        parsed["image_path"] = image_path
        results.append(parsed)

    return results


In [ ]:
# 05 | End-to-end insertion pipeline
# Run this cell again to produce a new independent experiment.
random.seed(SEED)
np.random.seed(SEED)
if SELECTION_MODE not in {"selection", "ranking"}:
    raise ValueError("SELECTION_MODE must be selection or ranking")
if not DIRECTIONS or any(d not in {"front", "back", "left", "right", "up", "down"} for d in DIRECTIONS):
    raise ValueError("Invalid attachment directions")
if not NEW_OBJECT_MESH_PATH.is_file():
    raise FileNotFoundError(NEW_OBJECT_MESH_PATH)
if not (0 < VOXEL_THRESHOLD <= 1 and 0 < INITIAL_SCALE_RADIUS < 1):
    raise ValueError("Invalid voxel threshold or scale radius")
if N_SCALE_CANDIDATES < 3 or N_SCALE_CANDIDATES % 2 == 0:
    raise ValueError("Use an odd number of scale candidates, at least 3")
if not (1 <= N_REFERENCE_VIEWS <= N_PART_VIEWS and 1 <= N_FITTING_VIEWS <= MAX_ANCHOR_VIEWS <= N_BASE_VIEWS):
    raise ValueError("Invalid view counts")
if min(N_SCALE_ITERATIONS, N_ROTATION_ITERATIONS, N_TRANSLATION_ITERATIONS) < 1:
    raise ValueError("All search stages must run at least once")

run_dir = OUTPUT_ROOT / f"{uid}_{datetime.now():%Y%m%d_%H%M%S_%f}"
run_dir.mkdir(parents=True, exist_ok=False)
request = f"Component: {COMPONENT_PROMPT}\nAttachment directions: {', '.join(DIRECTIONS)}\n{USER_PROMPT.strip()}"
config_snapshot = {key: (str(value) if isinstance(value, Path) else value)
                   for key, value in list(globals().items())
                   if key.isupper() and isinstance(value, (str, int, float, bool, list, Path))
                   and key not in {"FRONT_VIEW_SELECTION_SYSTEM_PROMPT", "ATTACHMENT_POINT_SYSTEM_PROMPT", "INSERTION_BOUNDING_BOX_SYSTEM_PROMPT", "RELATIVE_INSERTION_BBOX_SYSTEM_PROMPT"}}
(run_dir / "config.json").write_text(json.dumps(config_snapshot, indent=2), encoding="utf-8")
print(f"Run directory: {run_dir}")

# 1. Prepare meshes and render reference views.
log_stage("1 / 8 — Prepare and render meshes")
base_mesh_fit = ensure_trimesh(mesh)
if REMOVE_BASE_PLATFORM:
    base_mesh_fit = ensure_trimesh(_remove_platform_only(base_mesh_fit))
new_object_mesh = trimesh.load(str(NEW_OBJECT_MESH_PATH))
new_object_mesh_fit = ensure_trimesh(new_object_mesh)
base_o3d = to_o3d_mesh(base_mesh_fit)
part_o3d = to_o3d_mesh(new_object_mesh_fit)
base_bbox = base_o3d.get_axis_aligned_bounding_box()
base_center = base_bbox.get_center()
base_extent = float(base_bbox.get_max_extent())
if base_extent <= 0 or new_object_mesh_fit.extents.max() <= 0:
    raise ValueError("Mesh has zero extent")

rendered_info = render_n_views(
    base_o3d, n_views=N_BASE_VIEWS, output_dir=str(run_dir / "base_views"),
    width=BASE_IMAGE_SIZE, height=BASE_IMAGE_SIZE, frame_scale=FRAME_SCALE,
    is_objaverse_model=False
)
rendered_new_object_info = render_n_views(
    part_o3d, n_views=N_PART_VIEWS, output_dir=str(run_dir / "component_views"),
    width=BASE_IMAGE_SIZE, height=BASE_IMAGE_SIZE, frame_scale=PART_FRAME_SCALE,
    is_objaverse_model=False
)
views_grid = create_views_grid(rendered_info)
views_grid_path = run_dir / "base_views_grid.png"
views_grid.save(views_grid_path)
part_grid = create_views_grid(rendered_new_object_info, cols=min(6, N_PART_VIEWS))
part_grid.save(run_dir / "component_views_grid.png")
if SHOW_INTERMEDIATE:
    display(views_grid, part_grid)

# 2. Semantic front, object-centric axes, relevant cameras.
log_stage("2 / 8 — Semantic orientation and relevant views")
orientation = estimate_object_orientation(
    base_mesh_fit, rendered_info, grid_path=str(views_grid_path),
    ollama_host=OLLAMA_HOST, model_name=MODEL_NAME
)
classified_views = classify_views(rendered_info, orientation)
selected_views = select_relevant_views(classified_views, DIRECTIONS,
                                       threshold=VIEW_THRESHOLD, max_views=MAX_ANCHOR_VIEWS)
print("Front view:", orientation["front_view_idx"] + 1)
print("Relevant views:", [i + 1 for i in selected_views])
if len(selected_views) == 0:
    raise ValueError("No relevant cameras found")

# 3. Predict attachment points; select a real surface sample from the largest cluster.
log_stage("3 / 8 — Attachment point prediction")
anchor_points, anchor_normals = [], []
for i in selected_views:
    eye_pos, view_path = rendered_info[i]
    point_2d = generate_attachment_point(
        ollama_host=OLLAMA_HOST, image_path=view_path,
        rendered_new_object_info=rendered_new_object_info,
        user_prompt=request, model_name=MODEL_NAME
    )
    if point_2d is None:
        print(f"View {i + 1}: VLM rejected")
        continue
    if SHOW_INTERMEDIATE:
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(Image.open(view_path)); ax.scatter(*point_2d, c="red", s=60)
        ax.set_title(f"Attachment point — view {i + 1}"); ax.axis("off")
        plt.show(); plt.close(fig)
    point_3d, normal = pixel_to_3d(base_mesh_fit, point_2d, eye_pos,
                                    width=BASE_IMAGE_SIZE, height=BASE_IMAGE_SIZE,
                                    frame_scale=FRAME_SCALE)
    if point_3d is None:
        print(f"View {i + 1}: ray missed")
        continue
    anchor_points.append(point_3d)
    anchor_normals.append(normal)
    print(f"View {i + 1}: {point_3d}")
anchor_points = np.asarray(anchor_points)
anchor_normals = np.asarray(anchor_normals)
anchor_point, anchor_normal = select_anchor_cluster(
    anchor_points, anchor_normals, base_extent, ANCHOR_CLUSTER_RADIUS, ANCHOR_MIN_SAMPLES
)
if np.linalg.norm(anchor_normal) < 1e-8:
    raise ValueError("Invalid anchor normal")
print("Anchor:", anchor_point, "Normal:", anchor_normal)

# 4. Anchor-marked views and relative bounding boxes (absolute image coordinates).
log_stage("4 / 8 — Bounding box prediction")
anchor_rendered_info = rerender_scene_with_anchor(
    mesh=base_mesh_fit, anchor_point=anchor_point, rendered_info=rendered_info,
    output_dir=str(run_dir / "anchor_views"), width=BASE_IMAGE_SIZE, height=BASE_IMAGE_SIZE,
    frame_scale=FRAME_SCALE, marker_radius_ratio=MARKER_RADIUS_RATIO
)
# Anchor rendering uses combined base + marker bounds; match them for projection.
marker = o3d.geometry.TriangleMesh.create_sphere(radius=base_extent * MARKER_RADIUS_RATIO)
marker.translate(anchor_point)
render_bbox = (base_o3d + marker).get_axis_aligned_bounding_box()
render_center = render_bbox.get_center()
render_extent = render_bbox.get_max_extent()
inserted_object_image_paths = [path for _, path in rendered_new_object_info[:N_REFERENCE_VIEWS]]
relative_bbox_results = []
for i, (eye_pos, view_path) in enumerate(anchor_rendered_info):
    camera_up = _auto_camera_up(base_center, eye_pos)
    anchor_2d = project_points_to_image(
        np.array([anchor_point]), eye_pos, render_center, render_extent,
        width=BASE_IMAGE_SIZE, height=BASE_IMAGE_SIZE, frame_scale=FRAME_SCALE, up=camera_up
    )[0]
    if not is_anchor_visible(view_path, anchor_2d, radius=ANCHOR_VISIBILITY_RADIUS):
        print(f"View {i + 1:03d}: anchor not visible")
        continue
    result = generate_relative_insertion_bbox(
        ollama_host=OLLAMA_HOST, image_path=view_path, reference_images=inserted_object_image_paths,
        anchor_2d=anchor_2d, user_prompt=request, model_name=MODEL_NAME
    )
    result.update(view_idx=i, eye_pos=eye_pos, image_path=view_path, up=camera_up, anchor_2d=anchor_2d)
    relative_bbox_results.append(result)
    print(f"View {i + 1:03d}: {result['bbox_2d']}")
    if SHOW_INTERMEDIATE:
        visualize_bbox_result(view_path, result)
print("Valid bounding boxes:", sum(r["is_found"] for r in relative_bbox_results))

# 5. Multi-view voxel voting and initial candidate transformation.
log_stage("5 / 8 — Voxel voting and initialization")
voxel_points, voxel_size = create_voxel_grid(base_mesh_fit, resolution=VOXEL_RESOLUTION, padding=VOXEL_PADDING)
relative_voxel_scores = vote_for_voxels(
    voxel_points, relative_bbox_results, render_center, render_extent,
    width=BASE_IMAGE_SIZE, height=BASE_IMAGE_SIZE, frame_scale=FRAME_SCALE
)
selected_voxels = voxel_points[relative_voxel_scores >= VOXEL_THRESHOLD]
if len(selected_voxels) == 0:
    raise ValueError("No voxels satisfy the voting threshold")
print(f"Selected voxels: {len(selected_voxels)} / {len(voxel_points)}")
np.savez_compressed(run_dir / "grounding.npz", anchor_point=anchor_point, anchor_normal=anchor_normal,
                    selected_voxels=selected_voxels, voxel_size=voxel_size)
if SHOW_3D:
    visualize_voxel_grid(base_mesh_fit, selected_voxels, anchor_point).show()
fitting_views = select_fitting_views(relative_bbox_results, selected_views, n_views=N_FITTING_VIEWS)
if len(fitting_views) < N_FITTING_VIEWS:
    raise ValueError(f"Not enough valid fitting views: {len(fitting_views)} / {N_FITTING_VIEWS}")
print("Fitting views:", [v["view_idx"] + 1 for v in fitting_views])
init_scale, base_rotation, base_translation, attachment_axis_local = initialize_scale_transform(
    base_mesh_fit, new_object_mesh_fit, selected_voxels, anchor_point, anchor_normal,
    penetration_ratio=PENETRATION_RATIO
)
initial_mesh = transform_object_mesh(new_object_mesh_fit, init_scale, base_rotation, base_translation)
print("Initial scale:", init_scale)
if SHOW_3D:
    trimesh.Scene([base_mesh_fit, initial_mesh]).show()

# Shared fitting settings: all stages see the same cameras and image layout.
search_options = dict(ollama_host=OLLAMA_HOST, model_name=FITTING_MODEL, mode=SELECTION_MODE,
                      image_size=FITTING_IMAGE_SIZE, frame_scale=FITTING_FRAME_SCALE,
                      show_intermediate=SHOW_INTERMEDIATE)

# 6. Scale: update size AND pivot-compensated translation.
log_stage("6 / 8 — Scale search")
scale_history = run_scale_search(
    base_mesh=base_mesh_fit, new_object_mesh=new_object_mesh_fit, fitting_views=fitting_views,
    base_rotation=base_rotation, base_translation=base_translation, init_scale=init_scale,
    anchor_point=anchor_point, user_prompt=request, output_root=str(run_dir / "scale_search"),
    n_iterations=N_SCALE_ITERATIONS, n_candidates=N_SCALE_CANDIDATES,
    initial_radius=INITIAL_SCALE_RADIUS, **search_options
)
best_scale_candidate = scale_history[-1]["selected_candidate"]
best_scale = best_scale_candidate["scale"]
best_scale_rotation = best_scale_candidate["rotation"].copy()
best_scale_translation = best_scale_candidate["translation"].copy()
best_scale_mesh = best_scale_candidate["mesh"]
if SHOW_3D:
    trimesh.Scene([base_mesh_fit, best_scale_mesh]).show()

# 7. Rotation: start from the complete selected scale transformation.
log_stage("7 / 8 — Rotation search")
rotation_history = run_rotation_search(
    base_mesh=base_mesh_fit, new_object_mesh=new_object_mesh_fit, fitting_views=fitting_views,
    anchor_point=anchor_point, best_scale=best_scale, base_rotation=best_scale_rotation,
    base_translation=best_scale_translation, user_prompt=request,
    output_root=str(run_dir / "rotation_search"), n_iterations=N_ROTATION_ITERATIONS,
    initial_angle_step=INITIAL_ANGLE_STEP, **search_options
)
best_rotation_candidate = rotation_history[-1]["selected_candidate"]
best_rotation = best_rotation_candidate["rotation"].copy()
best_rotation_translation = best_rotation_candidate["translation"].copy()
best_rotation_mesh = best_rotation_candidate["mesh"]
if SHOW_3D:
    trimesh.Scene([base_mesh_fit, best_rotation_mesh]).show()

# 8. Translation: keep the selected scale and rotation unchanged.
log_stage("8 / 8 — Translation search and export")
translation_history = run_translation_search(
    base_mesh=base_mesh_fit, new_object_mesh=new_object_mesh_fit, fitting_views=fitting_views,
    anchor_point=anchor_point, anchor_normal=anchor_normal, best_scale=best_scale,
    base_rotation=best_rotation, base_translation=best_rotation_translation,
    object_front=orientation["front"], user_prompt=request,
    output_root=str(run_dir / "translation_search"), n_iterations=N_TRANSLATION_ITERATIONS,
    initial_step=base_extent * INITIAL_TRANSLATION_RATIO, **search_options
)
best_translation_candidate = translation_history[-1]["selected_candidate"]
best_translation = best_translation_candidate["translation"].copy()
best_translation_mesh = best_translation_candidate["mesh"]

# Apply the final transform once to the ORIGINAL inserted mesh.
final_scale = best_scale
final_rotation = best_rotation.copy()
final_translation = best_translation.copy()
final_inserted_mesh = transform_object_mesh(new_object_mesh_fit, final_scale, final_rotation, final_translation)
assert np.allclose(final_inserted_mesh.vertices, best_translation_mesh.vertices), "Final transform differs from VLM selection"
final_scene = trimesh.Scene([base_mesh_fit, final_inserted_mesh])
final_transform = np.eye(4)
final_transform[:3, :3] = final_scale * final_rotation
final_transform[:3, 3] = final_translation

# Keep decisions and exact transforms for later model comparisons.
for name, history in [("scale", scale_history), ("rotation", rotation_history), ("translation", translation_history)]:
    records = []
    for step in history:
        record = {key: value for key, value in step.items() if key not in {"candidate_infos", "selected_candidate"}}
        record["candidates"] = [
            {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in candidate.items() if k != "mesh"}
            for candidate in step["candidate_infos"]
        ]
        record["selected_candidate"] = {
            k: (v.tolist() if isinstance(v, np.ndarray) else v)
            for k, v in step["selected_candidate"].items() if k != "mesh"
        }
        records.append(record)
    (run_dir / f"{name}_history.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
(run_dir / "final_transform.json").write_text(json.dumps({
    "scale": float(final_scale), "rotation": final_rotation.tolist(),
    "translation": final_translation.tolist(), "matrix": final_transform.tolist()
}, indent=2), encoding="utf-8")
if EXPORT_GLB:
    final_scene.export(str(run_dir / "final_edited_object.glb"))
final_view_paths = render_scene_views_with_object(
    base_mesh_fit, final_inserted_mesh, fitting_views, str(run_dir / "final_views"),
    image_size=FITTING_IMAGE_SIZE, frame_scale=FITTING_FRAME_SCALE
)
final_collage_path = build_candidate_collage(final_view_paths, str(run_dir / "final_collage.png"))
show_image(final_collage_path)
if SHOW_3D:
    final_scene.show()
print("\nDone. Final scale:", final_scale)
print("Final rotation:\n", final_rotation)
print("Final translation:", final_translation)
print("Results:", run_dir.resolve())
